<a href="https://colab.research.google.com/github/vignesh-potharaj/StudyBot/blob/main/studybot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [17]:
#Cell 1 — install
!pip install transformers torch --quiet


In [ ]:
# Cell 2 — imports

from transformers import AutoTokenizer, AutoModel
import torch
import numpy as np

In [ ]:
# Cell 3 — load tokeniser and model
# output_attentions=True tells the model to return attention tensors
# instead of computing them internally and discarding them
MODEL_NAME = "distilbert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModel.from_pretrained(MODEL_NAME, output_attentions=True)
model.eval() # inference mode: no dropout, no gradient tracking
print("Model loaded:", MODEL_NAME)

In [ ]:
# Cell 4 — tokenise the study question
# return_tensors="pt" = PyTorch tensors
# add_special_tokens=True adds [CLS] at start, [SEP] at end
question = "Explain Newton's third law of motion."
inputs = tokenizer(question, return_tensors="pt", add_special_tokens=True)
token_ids = inputs["input_ids"][0]
tokens = tokenizer.convert_ids_to_tokens(token_ids)
print("Tokens:", tokens)


In [ ]:
# Cell 5 — forward pass
# torch.no_grad() disables gradient computation (saves memory during inference)
# outputs.attentions is a tuple of 6 tensors, one per transformer layer
with torch.no_grad():
 outputs = model(**inputs)
print("Number of attention layers:", len(outputs.attentions)) # 6
print("Layer 0 shape:", outputs.attentions[0].shape)
# torch.Size([1, 12, 11, 11])
# (batch=1, heads=12, from_token=11, to_token=11)


In [ ]:
def top_attention_pairs(model_outputs, tokens, layer, head, k=5):
    """
    Safely print the top-k attention pairs by explicitly passing dependencies.
    """
    # Ensure attentions exist
    if model_outputs.attentions is None:
        raise ValueError("Model was not loaded with output_attentions=True")

    attn = model_outputs.attentions[layer][0][head].numpy()
    flat = attn.flatten()
    top_idx = flat.argsort()[::-1][:k]

    # Get the matrix dimensions explicitly
    num_rows, num_cols = attn.shape

    print(f"\nLayer {layer}, Head {head}")
    print(f"{'FROM':<14} {'TO':<14} {'WEIGHT':>8}")
    print("-" * 40)
    for idx in top_idx:
        row = idx // num_cols
        col = idx % num_cols
        print(f"{tokens[row]:<14} {tokens[col]:<14} {flat[idx]:.3f}")

# Call it by passing the actual variables:
top_attention_pairs(outputs, tokens, layer=0, head=0, k=5)

In [ ]:
# Cell 7 — visualise attention as a heatmap (bonus)
import matplotlib.pyplot as plt
layer, head = 0, 0
attn = outputs.attentions[layer][0][head].numpy()
fig, ax = plt.subplots(figsize=(8, 6))
im = ax.imshow(attn, cmap="Blues", vmin=0, vmax=attn.max())
ax.set_xticks(range(len(tokens))); ax.set_xticklabels(tokens, rotation=45,
ha="right")
ax.set_yticks(range(len(tokens))); ax.set_yticklabels(tokens)
ax.set_title(f"Attention weights — Layer {layer}, Head {head}")
plt.colorbar(im, ax=ax)
plt.tight_layout()
plt.show()

In [ ]:
# Install the modern official Google GenAI package
!pip install google-genai --quiet

import os
from google import genai
from google.genai import types

In [ ]:
SYSTEM_PROMPTS = {

    "beginner": """You are StudyBot, a friendly tutor for first-year students.
Rules:
- Use only everyday words. Avoid jargon. If you must use a technical term, define it immediately.
- Use a simple real-world analogy first, then explain the concept.
- Keep your answer to 3–4 sentences maximum.
- End with one encouraging sentence.
Example of your tone: 'Great question! Think of it like...'
""",

    "intermediate": """You are StudyBot, a concise engineering tutor for second or third-year students.
Rules:
- Assume the student knows basic calculus and Python.
- Explain the concept, then show the key formula, then a one-paragraph worked example.
- Use bullet points for the algorithm steps.
- Keep the total response under 200 words.
""",

    "expert": """You are StudyBot, a graduate-level AI research assistant.
Rules:
- The student is a final-year or postgraduate student. Skip basic definitions.
- Discuss convergence properties, common failure modes, and practical trade-offs.
- Compare at least two variants (e.g. SGD vs Adam) with their mathematical motivation.
- Include pseudocode if relevant.
- Use precise mathematical notation.
"""
}

In [ ]:
# ── Updated Cell 3: Initialize the Gemini client safely ──────────────────────────
from google.colab import userdata

# 1. Fetch the secret key directly from your Colab Secrets
try:
    api_key = userdata.get('GEMINI_API_KEY')
except userdata.SecretNotFoundError:
    # Fallback just in case the name has a slight typo in your secrets panel
    api_key = userdata.get('GEMINI')

# 2. Explicitly hand the key to the Gemini Client
client = genai.Client(api_key=api_key)

def ask_studybot(question: str, level: str) -> str:
    """Send the question with the appropriate Gemini system instruction configuration."""
    config = types.GenerateContentConfig(
        system_instruction=SYSTEM_PROMPTS[level],
        max_output_tokens=512
    )

    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=question,
        config=config
    )

    return response.text

In [ ]:
question = "What is gradient descent?"

responses = {}
for level in ["beginner", "intermediate", "expert"]:
    print(f"\n{'='*60}")
    print(f"LEVEL: {level.upper()}")
    print('='*60)
    responses[level] = ask_studybot(question, level)
    print(responses[level])

In [ ]:
import re

def word_count(text):
    return len(text.split())

def avg_word_len(text):
    words = re.findall(r'\b\w+\b', text)
    return round(sum(len(w) for w in words)/len(words), 1) if words else 0

print(f"\n{'Metric':<25} {'Beginner':>12} {'Intermediate':>14} {'Expert':>10}")
print("-" * 65)
for key, label, fn in [
    ("word_count",    "Word count",        word_count),
    ("avg_word_len",  "Avg word length",   avg_word_len),
]:
    row = {lvl: fn(responses[lvl]) for lvl in ["beginner","intermediate","expert"]}
    print(f"{label:<25} {row['beginner']:>12} {row['intermediate']:>14} {row['expert']:>10}")

In [ ]:
# ── Mission 3: Cell 1 ──────────────────────────────────────────────────────────
!pip install faiss-cpu sentence-transformers google-genai --quiet

from sentence_transformers import SentenceTransformer
import faiss
import numpy as np
from google import genai
from google.genai import types
from google.colab import userdata

In [ ]:
# ── Mission 3: Cell 2 ──────────────────────────────────────────────────────────
LECTURE_NOTES = [
    {
        "title": "Newton's Second Law",
        "text": "Newton's second law states that the net force on an object equals its mass "
                "multiplied by its acceleration: F = ma. If multiple forces act on an object, "
                "you sum them as vectors. A 10 kg box accelerated at 3 m/s² requires a net "
                "force of 30 N. The law holds in inertial reference frames only."
    },
    {
        "title": "Conservation of Energy",
        "text": "The law of conservation of energy states that energy cannot be created or "
                "destroyed, only transformed. In a closed system, total mechanical energy "
                "(kinetic + potential) remains constant if only conservative forces act. "
                "KE = 0.5 * m * v², PE = m * g * h."
    },
    {
        "title": "Ohm's Law and Circuits",
        "text": "Ohm's law relates voltage, current, and resistance: V = IR. In series "
                "circuits, resistances add: R_total = R1 + R2 + ... In parallel circuits, "
                "1/R_total = 1/R1 + 1/R2 + ... Power dissipated: P = IV = I²R = V²/R."
    },
    {
        "title": "Gradient Descent",
        "text": "Gradient descent minimises a loss function L(θ) by updating parameters "
                "in the direction of the negative gradient: θ = θ - α * ∇L(θ). The learning "
                "rate α controls step size. Too large → divergence. Too small → slow convergence. "
                "Stochastic gradient descent (SGD) uses one sample per update; mini-batch uses a subset."
    },
    {
        "title": "Big-O Notation",
        "text": "Big-O notation describes algorithm time complexity as input size n grows. "
                "O(1) = constant time. O(log n) = logarithmic (binary search). O(n) = linear scan. "
                "O(n log n) = efficient sorting (merge sort). O(n²) = nested loops. "
                "We always take the dominant term and drop constants."
    },
    {
        "title": "Thermodynamics — First Law",
        "text": "The first law of thermodynamics is a statement of energy conservation for "
                "thermodynamic systems: ΔU = Q - W, where ΔU is the change in internal energy, "
                "Q is heat added to the system, and W is work done by the system. "
                "For an ideal gas, internal energy depends only on temperature."
    },
]

print(f"Knowledge base: {len(LECTURE_NOTES)} chunks loaded")

In [ ]:
#Cell 3 ──────────────────────────────────────────────────────────
embedder = SentenceTransformer("all-MiniLM-L6-v2")
texts      = [chunk["text"] for chunk in LECTURE_NOTES]
embeddings = embedder.encode(texts, show_progress_bar=True)

In [ ]:
#Cell 4 ──────────────────────────────────────────────────────────
dimension = embeddings.shape[1]
index     = faiss.IndexFlatL2(dimension)
index.add(np.array(embeddings, dtype="float32"))

In [ ]:
#Cell 5 ──────────────────────────────────────────────────────────
def retrieve(query: str, k: int = 2) -> list[dict]:
    query_vec = embedder.encode([query])
    distances, indices = index.search(np.array(query_vec, dtype="float32"), k)
    results = []
    for dist, idx in zip(distances[0], indices[0]):
        results.append({
            "title":    LECTURE_NOTES[idx]["title"],
            "text":     LECTURE_NOTES[idx]["text"],
            "distance": round(float(dist), 3)
        })
    return results

In [ ]:
#Cell 6 (Updated for Gemini) ──────────────────────────────────────

# Initialize client using your validated safe Colab Secrets setup
api_key = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=api_key)

def rag_answer(question: str) -> str:
    """Retrieve relevant notes then generate a grounded answer using Gemini."""

    # Step 1: Retrieve matching materials from vector index
    chunks = retrieve(question, k=2)
    context = "\n\n".join(f"[{c['title']}]\n{c['text']}" for c in chunks)

    # Step 2: Build the augmented prompt constraining Gemini to the source document
    prompt = f"""You are StudyBot. Answer the student's question using ONLY the
provided lecture notes. If the notes do not contain enough information,
say so — do not use outside knowledge.

LECTURE NOTES:
{context}

STUDENT QUESTION:
{question}

ANSWER:"""

    # Step 3: Generate response via gemini-2.5-flash
    response = client.models.generate_content(
        model="gemini-2.5-flash",
        contents=prompt
    )

    return response.text

# Test Run the code
question = "What formula should I use to find force when I know mass and acceleration?"
print("Query:", question)
print("\nRetrieved context chunks:")
for c in retrieve(question, k=2):
    print(f"  [{c['title']}]")

print("\nStudyBot Gemini answer:")
print(rag_answer(question))